# Backbone comparisons

goal:
1. smaller gap between Direct-Hybrid
2. Hybrid win Pop

ablation:
- regressor: ridge (เคาะ)
- support size: 10 25 50 100
- Mediator: Population, Direct, Hybrid
- Anchor: B, C
    | Anchor | how the personal head is anchored | anchored to |
    |---|---|---|
    | **B** | shrink the weights toward `w_pop` | the population fit **in that mediator's own space** |
    | **C** | fit on the residual `y - y_pop` | the **same true GIAA model** for every mediator |

    note: B each mediator is anchored to a different baseline, so Direct (anchored to the full 512-d GIAA) and Hybrid (anchored to a weaker 7-d one) do not start from the same place. 
    C is the fair comparison.


Everything is from `output/raw_all.csv`

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

from src.utils.metrics import tost_equivalence

LABEL = {'clip': 'CLIP frozen', 'clip_ft': 'CLIP-ft (score)',
         'clip_ft_emo': 'CLIP-ft (emotion)', 'qwen4b': 'Qwen3-VL 4B',
         'qwen8b': 'Qwen3-VL 8B'}
ORDER = list(LABEL)
UNIT = ['fold', 'domain', 'user_id']

ALL = pd.read_csv('../output/raw_all_final.csv', low_memory=False)
ALL = ALL[ALL['head'] == 'ridge']
raw = ALL[ALL['n_train'] == 100]
print(f'{len(raw):,} rows at n=100 | backbones: {sorted(raw.backbone.unique())}')


# One backbone across every support size and anchor

- `gap` is Direct - Hybrid
- `H - Pop` is Hybrid - population
- `p` is the paired Wilcoxon of Hybrid against population across the 387 user-domain units.

An anchor that has not been run for a backbone is reported as missing.


In [ ]:
ANCHORS = ['plain', 'A', 'B', 'C']
SIZES = [10, 25, 50, 100]


def by_support(backbone, variant):
    rows = []
    for n in SIZES:
        p = per_unit(ALL, backbone, variant, n=n)
        if p is None:
            continue
        H, D, P = p['emotion'], p['identity'], p['population']
        rows.append({'n_train': n,
                     'Hybrid': H.mean(), 'Direct': D.mean(),
                     'population': P.mean(),
                     'gap': (D - H).mean(),
                     'H - Pop': (H - P).mean(),
                     'p': wilcoxon(H, P)[1],
                     'Hybrid wins': (H > P).mean()})
    return pd.DataFrame(rows).set_index('n_train') if rows else None


SFMT = {'Hybrid': '{:.4f}', 'Direct': '{:.4f}', 'population': '{:.4f}',
        'gap': '{:+.4f}', 'H - Pop': '{:+.4f}', 'p': '{:.4f}',
        'Hybrid wins': '{:.1%}'}


def anchor_panels(backbone):
    """One styled table per anchor, or a note saying what is missing."""
    from IPython.display import display, Markdown
    display(Markdown(f'### {LABEL[backbone]}'))
    for v in ANCHORS:
        t = by_support(backbone, v)
        if t is None:
            display(Markdown(f'- **anchor {v}** - not run yet'))
            continue
        missing = sorted(set(SIZES) - set(t.index))
        note = f' (missing n={missing})' if missing else ''
        display(t.style.format(SFMT)
                .background_gradient(subset=['H - Pop'], cmap='RdYlGn',
                                     vmin=-0.05, vmax=0.05)
                .apply(lambda s: ['font-weight:bold' if v_ < .05 else 'color:#999'
                                  for v_ in s], subset=['p'])
                .set_caption(f'{LABEL[backbone]} x ridge - anchor {v}{note}'))


## CLIP-ft (emotion)


In [ ]:
anchor_panels('clip_ft_emo')

## The other backbones, for reference


In [ ]:
for bb in ('clip_ft','clip', 'qwen4b', 'qwen8b'):
    anchor_panels(bb)

Reading it: 

under **plain**, the 7-d mediator is ahead at every small support
size on the frozen backbones, and the advantage decays as n grows -- low-data argument predicts. 

Under **C**, the population anchor already supplies what the mediator was supplying, so the remaining difference is the raw capacity of the correction and Direct leads throughout.

The two fine-tuned backbones behave differently under plain, because their
features are already task-aligned 


**Caveat that has to travel with the CLIP-ft (emotion) row**

That backbone was fine-tuned to predict the seven emotions directly, so its
512-d features are already an emotion representation. 'Direct' on it is not a
mediator-free control -- it is a 512-d emotion representation competing with a
7-d one, and the wider gap follows from that rather than from anything about
the mediator. Its gap should not be read against the other backbones'.


---
## Backbone comparisons, by user-domain unit

A *unit* is one user in one domain (387). Scores are averaged over the 3 seeds first, so each unit contributes once; 
the Wilcoxon test is then paired across units, which makes 'Hybrid beats population' a statement about users rather than about the mean.

In [ ]:
def per_unit(frame, backbone, variant, n=None):
    g = frame[(frame.backbone == backbone) & (frame.variant == variant)]
    if n is not None:
        g = g[g.n_train == n]
    if g.empty:
        return None
    g = g.groupby(UNIT + ['mediator'], as_index=False)['srocc'].mean()
    p = g.pivot_table(index=UNIT, columns='mediator', values='srocc')
    need = {'emotion', 'identity', 'population'}
    if not need <= set(p.columns):
        return None
    return p.dropna(subset=list(need))


def table(variant):
    rows = []
    for bb in ORDER:
        p = per_unit(raw, bb, variant)
        if p is None:
            continue
        H, D, P = p['emotion'], p['identity'], p['population']
        rows.append({'backbone': LABEL[bb], 'units': len(p),
                     'Population': P.mean(),
                     'Hybrid (7-d)': H.mean(),
                     'Direct (512-d)': D.mean(),
                     'Hybrid - Pop': (H - P).mean(),
                     'p (Wilcoxon)': wilcoxon(H, P)[1],
                     'Hybrid wins': (H > P).mean(),
                     'Direct - Hybrid gap': (D - H).mean()})
    return pd.DataFrame(rows).set_index('backbone')


FMT = {'Population': '{:.4f}', 'Hybrid (7-d)': '{:.4f}',
       'Direct (512-d)': '{:.4f}', 'Hybrid - Pop': '{:+.4f}',
       'p (Wilcoxon)': '{:.4f}', 'Hybrid wins': '{:.1%}',
       'Direct - Hybrid gap': '{:+.4f}'}


def show(df, caption):
    return (df.style.format(FMT)
            .background_gradient(subset=['Hybrid - Pop'], cmap='RdYlGn',
                                 vmin=-0.02, vmax=0.02)
            .apply(lambda s: ['font-weight:bold' if v < .05 else 'color:#999'
                              for v in s], subset=['p (Wilcoxon)'])
            .set_caption(caption))


#### Table 1 - Anchor B
Each mediator is anchored to the population model fitted *in its own space*.
#### Table 2 - Anchor C
Every mediator is anchored to the **same** true GIAA model.

In [ ]:
tB = table('B')
show(tB, 'Anchor B, n=100, 3 seeds, ridge head. Bold p < .05.')
print(f'Hybrid significantly beats population on {(tB["p (Wilcoxon)"] < .05)} of {len(tB)} backbones')

tC = table('C')
show(tC, 'Anchor C, n=100, 3 seeds, ridge head. Bold p < .05.')
print(f"Hybrid significantly beats population on {(tC['p (Wilcoxon)'] < .05)} of {len(tC)} backbones")

## Why should C rather than B

Four checks. The first and the last are the strong ones; the middle two are
supporting evidence, and check 2 in particular is a difference of degree.

#### 1. Fair Comparison Baseline
Direct is mathematically the same model under B and C. For the identity mediator the transform is the identity, so w_pop IS the true GIAA and the two residuals coincide.
- residual B: y - X_u * w_pop
- residual C: y - y^_pop ; y^_pop comes from X_u * w_pop

while Hybrid is not the same, because it is anchored to different population fits.
- residual B: y - E_u * w_pop 
- residual C: y - y^_pop,512 ; closer to the true GIAA residual than B is
so C is the fair comparison. 
#### 2. The Direct-Hybrid gap shrinks once both are anchored to the same model.
#### 3. Significantly outperforms Population
#### 4. B has no defined form for an MLP head: it shrinks the weight vector
   toward w_pop, and an MLP has no weight vector to shrink. C is a residual
   fit and applies to any head, so MLP rows can only exist under C.

In [ ]:
chk = pd.DataFrame({'Direct under B': tB['Direct (512-d)'],
                    'Direct under C': tC['Direct (512-d)'],
                    'Hybrid under B': tB['Hybrid (7-d)'],
                    'Hybrid under C': tC['Hybrid (7-d)']})
chk['Direct identical'] = np.isclose(chk['Direct under B'],
                                     chk['Direct under C'], atol=1e-12)
chk['Hybrid identical'] = np.isclose(chk['Hybrid under B'],
                                     chk['Hybrid under C'], atol=1e-12)
chk.round(4)


Direct identical on CLIP frozen is False because at Anchor B it  only train on seed 0, not because the two residuals differ. 

In [ ]:
# How often does Hybrid sit BELOW the population row it is anchored on?
for variant in ('B', 'C'):
    bad = []
    for bb in ORDER:
        for n in (10, 25, 50, 100):
            p = per_unit(ALL, bb, variant, n=n)
            if p is None:
                continue
            if p['emotion'].mean() < p['population'].mean():
                bad.append(f'{LABEL[bb]} n={n}')
    print(f'anchor {variant}: Hybrid below population in {len(bad)} cells'
          + (': ' + ', '.join(bad) if bad else ''))


In [ ]:
# gap between Direct and Hybrid on B n C
gap = pd.DataFrame({'gap under B': tB['Direct - Hybrid gap'],
                    'gap under C': tC['Direct - Hybrid gap']})
gap['shrinkage'] = 1 - gap['gap under C'] / gap['gap under B']
gap.style.format({'gap under B': '{:+.4f}', 'gap under C': '{:+.4f}',
                  'shrinkage': '{:.0%}'})


# Direct-Hybrid gap
The gap is a capacity difference by design, not evidence that the mediator discards useful signal.
so we ask equivalence questions rather than difference questions: 
*is the 7-parameter head close enough to the 512-parameter one to be worth its interpretability*

# TOST (Two One-Sided Test) for equivalence
it rejects 'the difference is at least delta' from both sides, so passing means the true difference lies inside (-delta, +delta).
**delta must be fixed on substantive grounds before looking at the output.**
Several values are shown to expose the sensitivity.

In [ ]:
eq = []
for bb in ORDER:
    for n in (10, 25, 50, 100):
        p = per_unit(ALL, bb, 'C', n=n)
        if p is None:
            continue
        row = {'backbone': LABEL[bb], 'n': n,
               'Hybrid - Direct': (p['emotion'] - p['identity']).mean()}
        for d in (0.01, 0.02, 0.03):
            t = tost_equivalence(p['emotion'], p['identity'], delta=d)
            row[f'equivalent (d={d})'] = 'yes' if t['equivalent'] else 'no'
        eq.append(row)

eq = pd.DataFrame(eq).set_index(['backbone', 'n'])
eq.style.format({'Hybrid - Direct': '{:+.4f}'})
